# Prompt Chaining — Oil & Gas Production Reporting

## Pattern
Prompt chaining decomposes a task into sequential LLM calls where **each step's output becomes the next step's input**. This is the right pattern when:
- The final output requires multiple distinct transformations
- Each transformation depends on the result of the previous one
- No single prompt can reliably do all steps at once

## O&G Use Case: Daily Production Report
Raw SCADA telemetry from wellheads arrives as unstructured text — a mix of sensor readings, units, and operator notes. Before it becomes a usable daily report it must pass through four distinct steps:

1. **Extract** — Pull structured metrics from raw sensor text
2. **Normalize** — Convert all values to standard field units (bbl/day, MMSCFD, psi)
3. **Flag** — Compare against expected production targets and flag deviations
4. **Summarize** — Generate an executive-ready daily production summary

Each step requires different reasoning. Trying to do all four in one prompt degrades quality — chaining keeps each step focused.

In [ ]:
%pip install openai -q

In [ ]:
import sys, os
try:
    # Running via Databricks (DABs or interactive) — resolve notebook directory
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    sys.path.insert(0, '/Workspace' + _nb_path.rsplit('/', 1)[0])
except Exception:
    sys.path.insert(0, '.')  # local development fallback

from util import llm_call

def chain(input_text: str, steps: list[dict]) -> list[dict]:
    """Run a sequence of LLM prompts, passing each output to the next step.
    
    Args:
        input_text: The initial input to the chain.
        steps: List of dicts with 'name' and 'prompt' keys.
                Each prompt receives the previous output appended.
    
    Returns:
        List of dicts with step name and output for each step.
    """
    results = []
    current = input_text
    
    for step in steps:
        print(f"\n{'='*60}")
        print(f"Step: {step['name']}")
        print('='*60)
        output = llm_call(f"{step['prompt']}\n\nInput:\n{current}")
        print(output)
        results.append({'step': step['name'], 'output': output})
        current = output
    
    return results

## Mock SCADA Input
This simulates raw telemetry text as it arrives from a field SCADA system — a mix of units, informal notation, and operator annotations.

In [ ]:
RAW_SCADA = """
FIELD: Permian Basin - Block 7 West
DATE: 2024-03-15  SHIFT: Day (06:00-18:00)
OPERATOR: J. Martinez

WELL A-14:
  Tubing pressure: 1,842 psi
  Casing pressure: 2,105 psi  
  Oil rate: 387 bbl/d
  Gas rate: 0.52 MMscf/d
  Water cut: 18%
  Choke size: 24/64
  Status: FLOWING

WELL A-17:
  Tubing pressure: 620 psi  
  Casing pressure: 890 psi
  Oil rate: 41 m3/day   <- note: meter recently switched to metric
  Gas rate: 18,500 Mscf/d   <- note: confirm units with gauge tech
  Water cut: 67%
  Choke size: 48/64
  Status: FLOWING, HIGH WATER

WELL B-03:
  Tubing pressure: 0 psi
  Casing pressure: 1,200 psi
  Oil rate: 0 bbl/d
  Gas rate: 0 MMscf/d
  Water cut: N/A
  Choke size: CLOSED
  Status: SHUT-IN, awaiting workover crew. Bean plug set 03/12.

WELL B-09:
  Tubing pressure: 3,210 psi
  Casing pressure: 3,180 psi
  Oil rate: 512 bbl/d
  Gas rate: 1.1 MMscf/d
  Water cut: 4%
  Choke size: 16/64
  Status: FLOWING

WELL C-22:
  Tubing pressure: 410 psi  <- LOW, down from 890 psi yesterday
  Casing pressure: 1,750 psi
  Oil rate: 198 bbl/d  <- down ~35% vs prior day
  Gas rate: 0.31 MMscf/d
  Water cut: 29%
  Choke size: 32/64
  Status: FLOWING, possible liquid loading
"""

print("Raw SCADA input loaded. Length:", len(RAW_SCADA), "characters")

## Define the Chain Steps
Four focused prompts — each does one job well.

In [ ]:
PRODUCTION_CHAIN = [
    {
        "name": "1. Extract structured metrics",
        "prompt": """You are a petroleum data engineer. Extract all well measurements from the raw SCADA text.
For each well produce a clean structured block with these fields exactly:
  WELL: <id>
  TUBING_PSI: <value or N/A>
  CASING_PSI: <value or N/A>
  OIL_RATE: <value> <unit>
  GAS_RATE: <value> <unit>
  WATER_CUT_PCT: <value or N/A>
  CHOKE: <value or CLOSED>
  STATUS: <value>
  NOTES: <any operator annotations or flags>

Do not interpret or correct data yet — extract exactly what is reported."""
    },
    {
        "name": "2. Normalize to standard units",
        "prompt": """You are a petroleum data engineer. Normalize all rate values to standard field units:
- Oil rates → bbl/day (1 m3 = 6.2898 bbl)
- Gas rates → MMscf/day (1 Mscf = 0.001 MMscf)
- Pressures → psi (already in psi, keep as-is)

For each well, show the original value, the conversion applied (if any), and the normalized value.
Flag any wells where units were ambiguous or required assumptions.
Output one block per well using the same field structure as the input."""
    },
    {
        "name": "3. Flag anomalies vs. targets",
        "prompt": """You are a production engineer. Review the normalized well data against these field targets for Block 7 West:

  Expected oil rates:  A-14: 400 bbl/d | A-17: 280 bbl/d | B-03: 350 bbl/d | B-09: 500 bbl/d | C-22: 300 bbl/d
  Expected GOR:        ~1,400 scf/bbl for this field
  High water cut threshold: >50% requires review
  Low tubing pressure threshold: <500 psi requires investigation

For each well assign one of: [ON TARGET] [UNDERPERFORMING] [SHUT-IN] [NEEDS INVESTIGATION]
List the specific deviation or concern for any well not marked ON TARGET.
Calculate total field oil production (bbl/day) from flowing wells only."""
    },
    {
        "name": "4. Generate executive summary",
        "prompt": """You are a production operations manager. Write a concise daily production summary for Block 7 West.

Format:
DAILY PRODUCTION SUMMARY — Block 7 West
Date: 2024-03-15

HEADLINE NUMBERS
  Total field oil: X bbl/day  (vs. target: 1,830 bbl/day)
  Flowing wells: X of 5
  Wells requiring action: X

WELL STATUS
  [Well-by-well one-liner: status and key number]

PRIORITY ACTIONS
  [Numbered list, most urgent first]

Keep it under 200 words. This goes to the asset manager — be direct, no jargon beyond standard O&G terms."""
    }
]

## Run the Chain

In [ ]:
results = chain(RAW_SCADA, PRODUCTION_CHAIN)

## Final Output — Executive Summary

In [ ]:
print(results[-1]['output'])

## Key Takeaways

**Why chain instead of a single prompt?**
- Step 1 (extract) requires faithful transcription — no interpretation
- Step 2 (normalize) requires unit arithmetic — a different reasoning mode
- Step 3 (flag) requires comparing against external targets — needs the normalized numbers as input
- Step 4 (summarize) requires synthesizing the flags into a narrative — needs the full picture first

A single prompt asked to do all four simultaneously tends to hallucinate conversions, miss anomalies, or produce summaries inconsistent with the underlying data.

**Where to go from here:**
- Replace mock SCADA text with real telemetry from your historian or PI system
- Add a Step 5 to auto-generate work order tickets for flagged wells
- Schedule as a Databricks Job to run at end of each shift